## 使用训练后的模型在验证集上进行推理

In [1]:
import os
import pathlib
import tqdm
import numpy as np
import torch
from matplotlib import pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from skimage.io import imsave
from torch.utils.data import DataLoader
import PIL.Image
import IPython.display as ipd

from dataset import BrainSegmentationDataset
from unet import UNet
from utils import dsc, gray2rgb, outline, largest_connected_component

### 输入参数

- `data_root_dir`: 数据集路径  
- `ckpt_path`: 训练后的模型参数文件路径


In [2]:
data_root_dir = './kaggle_3m'
ckpt_path     = './ckpt/260326-155648 lishangzhe_ckpt/unet.pt'

batch_size    = 32
device        = torch.device('cuda:0')

In [3]:
dataset = BrainSegmentationDataset(images_dir=data_root_dir, is_train=False,)
loader = DataLoader(dataset, batch_size=batch_size, drop_last=False, num_workers=1)

unet = UNet(in_channels=BrainSegmentationDataset.in_channels, out_channels=BrainSegmentationDataset.out_channels)
state_dict = torch.load(ckpt_path, map_location=device)
unet.load_state_dict(state_dict)
unet.eval()
unet.to(device)

x_list = []
y_pred_list = []
y_true_list = []

with torch.no_grad():
    for i, (x, y_true, g_inv) in enumerate(tqdm.tqdm(loader, desc='Inference')):
        x      = x     .to(device)
        y_true = y_true.to(device)

        y_pred = unet(x)

        x_np      = x     .detach().cpu().numpy()
        y_pred_np = y_pred.detach().cpu().numpy()
        y_true_np = y_true.detach().cpu().numpy()

        y_pred_list.extend([y_pred_np[s] for s in range(y_pred_np.shape[0])])
        y_true_list.extend([y_true_np[s] for s in range(y_true_np.shape[0])])
        x_list     .extend([x_np     [s] for s in range(x_np     .shape[0])])

# 数据后处理
volumes = {}
num_slices = np.bincount([p[0] for p in dataset.patient_slice_index])
index = 0
for p in range(len(num_slices)):
    volume_in = np.array(x_list[index : index + num_slices[p]])
    volume_pred = np.round(np.array(y_pred_list[index : index + num_slices[p]])).astype(int)
    volume_pred = largest_connected_component(volume_pred)
    volume_true = np.array(y_true_list[index : index + num_slices[p]])
    volumes[dataset.patients[p]] = (volume_in, volume_pred, volume_true)
    index += num_slices[p]


Reading 355 images from validation set ...


Inference: 100%|██████████| 12/12 [00:13<00:00,  1.14s/it]


In [4]:
samples_dir = pathlib.Path(ckpt_path).parent / 'samples'
samples_dir.mkdir(parents=True, exist_ok=True)
print(f'Saving samples to: {samples_dir}')

for p in tqdm.tqdm(volumes):
    x      = volumes[p][0]
    y_pred = volumes[p][1]
    y_true = volumes[p][2]
    for s in range(x.shape[0]):
        image = gray2rgb(x[s, 1])  # channel 1 is for FLAIR
        image = outline(image, y_pred[s, 0], color=[255, 0, 0])
        image = outline(image, y_true[s, 0], color=[0, 255, 0])
        filename = '{}-{}.png'.format(p, str(s).zfill(2))
        filepath = pathlib.Path(samples_dir, filename)
        imsave(filepath, image)


Saving samples to: ckpt/260326-155648 lishangzhe_ckpt/samples


 50%|█████     | 5/10 [00:06<00:07,  1.50s/it]/tmp/ipykernel_367/1991996061.py:15: UserWarning: /xuetangx/yufeng/brain-seg/ckpt/260326-155648 lishangzhe_ckpt/samples/TCGA_DU_6408_19860521-55.png is a low contrast image
  imsave(filepath, image)
100%|██████████| 10/10 [00:16<00:00,  1.65s/it]
